# Mini Project 5-5 Explore Hypothesis Testing

## Introduction

You work for an environmental think tank called Repair Our Air (ROA). ROA is formulating policy recommendations to improve the air quality in America, using the Environmental Protection Agency's Air Quality Index (AQI) to guide their decision making. An AQI value close to 0 signals "little to no" public health concern, while higher values are associated with increased risk to public health. 

They've tasked you with leveraging AQI data to help them prioritize their strategy for improving air quality in America.

ROA is considering the following decisions. For each, construct a hypothesis test and an accompanying visualization, using your results of that test to make a recommendation:

1. ROA is considering a metropolitan-focused approach. Within California, they want to know if the mean AQI in Los Angeles County is statistically different from the rest of California.
2. With limited resources, ROA has to choose between New York and Ohio for their next regional office. Does New York have a lower AQI than Ohio?
3. A new policy will affect those states with a mean AQI of 10 or greater. Will Michigan be affected by this new policy?

**Notes:**
1. For your analysis, you'll default to a 5% level of significance.
2. Throughout the lab, for two-sample t-tests, use Welch's t-test (i.e., setting the `equal_var` parameter to `False` in `scipy.stats.ttest_ind()`). This will account for the possibly unequal variances between the two groups in the comparison.

## Step 1: Imports

To proceed with your analysis, import `pandas` and `numpy`. To conduct your hypothesis testing, import `stats` from `scipy`.

#### Import Packages

In [28]:
# Import relevant packages
import pandas as pd
import numpy as np
from scipy import stats

You are also provided with a dataset with national Air Quality Index (AQI) measurements by state over time for this analysis. `Pandas` was used to import the file `c4_epa_air_quality.csv` as a dataframe named `aqi`. As shown in this cell, the dataset has been automatically loaded in for you. You do not need to download the .csv file, or provide more code, in order to access the dataset and proceed with this lab. Please continue with this activity by completing the following instructions.

**Note:** For purposes of your analysis, you can assume this data is randomly sampled from a larger population.

#### Load Dataset

In [7]:
# IMPORT YOUR DATA
air_quality = pd.read_csv('c4_epa_air_quality.csv', index_col=0)

## Step 2: Data Exploration

### Before proceeding to your deliverables, explore your datasets.

Use the following space to surface descriptive statistics about your data. In particular, explore whether you believe the research questions you were given are readily answerable with this data.

In [8]:
# Use head() to show a sample of data
air_quality.head()

,date_local,state_name,county_name,city_name,local_site_name,parameter_name,units_of_measure,arithmetic_mean,aqi
0,2018-01-01,Arizona,Maricopa,Buckeye,BUCKEYE,Carbon monoxide,Parts per million,0.473684,7
1,2018-01-01,Ohio,Belmont,Shadyside,Shadyside,Carbon monoxide,Parts per million,0.263158,5
2,2018-01-01,Wyoming,Teton,Not in a city,Yellowstone National Park - Old Faithful Snow ...,Carbon monoxide,Parts per million,0.111111,2
3,2018-01-01,Pennsylvania,Philadelphia,Philadelphia,North East Waste (NEW),Carbon monoxide,Parts per million,0.300000,3
4,2018-01-01,Iowa,Polk,Des Moines,CARPENTER,Carbon monoxide,Parts per million,0.215789,3


In [9]:
# check varibles
air_quality.info()

<class 'pandas.core.frame.DataFrame'>
Index: 260 entries, 0 to 259
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        260 non-null    object 
 1   state_name        260 non-null    object 
 2   county_name       260 non-null    object 
 3   city_name         260 non-null    object 
 4   local_site_name   257 non-null    object 
 5   parameter_name    260 non-null    object 
 6   units_of_measure  260 non-null    object 
 7   arithmetic_mean   260 non-null    float64
 8   aqi               260 non-null    int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 20.3+ KB


In [10]:
# Use describe() to summarize AQI
air_quality['aqi'].describe()

count    260.000000
mean       6.757692
std        7.061707
min        0.000000
25%        2.000000
50%        5.000000
75%        9.000000
max       50.000000
Name: aqi, dtype: float64

In [11]:
# For a more thorough examination of observations by state use values_counts()
air_quality['state_name'].value_counts()

state_name
California              66
Arizona                 14
Ohio                    12
Florida                 12
Texas                   10
New York                10
Pennsylvania            10
Michigan                 9
Colorado                 9
Minnesota                7
New Jersey               6
Indiana                  5
North Carolina           4
Massachusetts            4
Maryland                 4
Oklahoma                 4
Virginia                 4
Nevada                   4
Connecticut              4
Kentucky                 3
Missouri                 3
Wyoming                  3
Iowa                     3
Hawaii                   3
Utah                     3
Vermont                  3
Illinois                 3
New Hampshire            2
District Of Columbia     2
New Mexico               2
Montana                  2
Oregon                   2
Alaska                   2
Georgia                  2
Washington               2
Idaho                    2
Nebraska         

#### **Question 1: From the preceding data exploration, what do you recognize?**

A: There are 9 columns in the data set. local_site_name columns has 3 missing values, aqi has no missing values, with min 0, max 50, and mean 6.76. The state appearance frequencies in the data are not the same, California appeared the most.

## Step 3. Statistical Tests

Before you proceed, recall the following steps for conducting hypothesis testing:

1. Formulate the null hypothesis and the alternative hypothesis.<br>
2. Set the significance level.<br>
3. Determine the appropriate test procedure.<br>
4. Compute the p-value.<br>
5. Draw your conclusion.

### Hypothesis 1: ROA is considering a metropolitan-focused approach. Within California, they want to know if the mean AQI in Los Angeles County is statistically different from the rest of California.

Before proceeding with your analysis, it will be helpful to subset the data for your comparison.

In [17]:
# Create dataframes for each sample being compared in your test
LA_air_quality = air_quality[(air_quality['state_name']=='California') & (air_quality['county_name']=='Los Angeles')]

CA_other_air_quality = air_quality[(air_quality['state_name']=='California') & (air_quality['county_name']!='Los Angeles')]


In [18]:
# Check head
LA_air_quality.head()


,date_local,state_name,county_name,city_name,local_site_name,parameter_name,units_of_measure,arithmetic_mean,aqi
33,2018-01-01,California,Los Angeles,Lancaster,Lancaster-Division Street,Carbon monoxide,Parts per million,0.394737,7
42,2018-01-01,California,Los Angeles,Santa Clarita,Santa Clarita,Carbon monoxide,Parts per million,0.394737,7
61,2018-01-01,California,Los Angeles,Pasadena,Pasadena,Carbon monoxide,Parts per million,0.789474,16
76,2018-01-01,California,Los Angeles,Los Angeles,LAX Hastings,Carbon monoxide,Parts per million,0.863158,17
109,2018-01-01,California,Los Angeles,Los Angeles,Los Angeles-North Main Street,Carbon monoxide,Parts per million,0.994737,17


In [19]:
CA_other_air_quality.head()

,date_local,state_name,county_name,city_name,local_site_name,parameter_name,units_of_measure,arithmetic_mean,aqi
16,2018-01-01,California,San Bernardino,Ontario,Ontario Near Road (Etiwanda),Carbon monoxide,Parts per million,0.747368,11
18,2018-01-01,California,Sacramento,Arden-Arcade,Sacramento-Del Paso Manor,Carbon monoxide,Parts per million,0.752632,16
26,2018-01-01,California,Orange,La Habra,La Habra,Carbon monoxide,Parts per million,0.673684,13
27,2018-01-01,California,Alameda,Not in a city,Berkeley- Aquatic Park,Carbon monoxide,Parts per million,1.088889,15
34,2018-01-01,California,Fresno,Fresno,Fresno - Garland,Carbon monoxide,Parts per million,1.000000,15


#### Formulate your hypothesis:

**Formulate your null and alternative hypotheses:**

*   $H_0$: There is no difference in the mean AQI between Los Angeles County and the rest of California.
*   $H_A$: There is a difference in the mean AQI between Los Angeles County and the rest of California.


#### Set the significance level:

For this analysis, the significance level is 5%

#### Determine the appropriate test procedure:

In [ ]:
LA_air_quality.info() #14
CA_other_air_quality.info() #52

<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, 33 to 250
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        14 non-null     object 
 1   state_name        14 non-null     object 
 2   county_name       14 non-null     object 
 3   city_name         14 non-null     object 
 4   local_site_name   14 non-null     object 
 5   parameter_name    14 non-null     object 
 6   units_of_measure  14 non-null     object 
 7   arithmetic_mean   14 non-null     float64
 8   aqi               14 non-null     int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 1.1+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 52 entries, 16 to 249
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        52 non-null     object 
 1   state_name        52 non-null     object 
 2   county_name       52 non-null 

Here, you are comparing the sample means between two independent samples. Therefore, you will utilize a **two-sample  𝑡-test**.

#### Compute the P-value

In [32]:
# Compute your p-value here
h1_result = stats.ttest_ind(a=LA_air_quality['aqi'], b=CA_other_air_quality['aqi'], equal_var=False)
h1_result

TtestResult(statistic=np.float64(2.1107010796372014), pvalue=np.float64(0.049839056842410995), df=np.float64(17.08246830361151))

#### **Question 2. What is your P-value for hypothesis 1, and what does this indicate for your null hypothesis?**

In [35]:
# Extracting pvalue and make the test
p_value = h1_result[1]
p_value

np.float64(0.049839056842410995)

A: The p-value is 0.0498, which is smaller than 0.05, our significance level, indicating we can reject the null hypothesis, and suggesting there is a difference in the mean AQI between Los Angeles County and the rest of California.

### Hypothesis 2: With limited resources, ROA has to choose between New York and Ohio for their next regional office. Does New York have a lower AQI than Ohio?

Before proceeding with your analysis, it will be helpful to subset the data for your comparison.

In [38]:
# Create dataframes for each sample being compared in your test
ohio_air_quality = air_quality[air_quality['state_name']=='Ohio']

ny_air_quality = air_quality[air_quality['state_name'] == 'New York']

In [ ]:
# Check head
ohio_air_quality.head()
ohio_air_quality.info() #12
ny_air_quality.head()
ny_air_quality.info() #10

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 1 to 252
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        12 non-null     object 
 1   state_name        12 non-null     object 
 2   county_name       12 non-null     object 
 3   city_name         12 non-null     object 
 4   local_site_name   12 non-null     object 
 5   parameter_name    12 non-null     object 
 6   units_of_measure  12 non-null     object 
 7   arithmetic_mean   12 non-null     float64
 8   aqi               12 non-null     int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 960.0+ bytes
<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 90 to 234
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        10 non-null     object 
 1   state_name        10 non-null     object 
 2   county_name       10 non-n

**Formulate your null and alternative hypotheses:**

*   $H_0$: The mean AQI of New York is greater than or equal to that of Ohio.
*   $H_A$: The mean AQI of New York is **below** that of Ohio.


#### Significance Level (remains at 5%)

#### Determine the appropriate test procedure:

Here, you are comparing the sample means between two independent samples in one direction. Therefore, you will utilize a **two-sample  𝑡-test**.

#### Compute the P-value

In [46]:
# Compute your p-value here
h2_result = stats.ttest_ind(ny_air_quality['aqi'], ohio_air_quality['aqi'], alternative='less')
print(h2_result)

TtestResult(statistic=np.float64(-1.891850434703295), pvalue=np.float64(0.03654034300840755), df=np.float64(20.0))


#### **Question 3. What is your P-value for hypothesis 2, and what does this indicate for your null hypothesis?**

In [47]:
# Your code here.
p_value_h2 = h2_result[1]
print(p_value_h2)

0.03654034300840755


A: The p-value for hypothesis 2 is 0.0365, which is less than our significance level, 0.05, indicating that we can reject the null hypothesis, and the mean AQI of New York is below that of Ohio.

###  Hypothesis 3: A new policy will affect those states with a mean AQI of 10 or greater. Will Michigan be affected by this new policy?

Before proceeding with your analysis, it will be helpful to subset the data for your comparison.

In [ ]:
# Create dataframes for each sample being compared in your test
MI_air_quality = air_quality[air_quality['state_name'] == "Michigan"]
MI_air_quality.head()
MI_air_quality.info() #9

<class 'pandas.core.frame.DataFrame'>
Index: 9 entries, 65 to 248
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date_local        9 non-null      object 
 1   state_name        9 non-null      object 
 2   county_name       9 non-null      object 
 3   city_name         9 non-null      object 
 4   local_site_name   9 non-null      object 
 5   parameter_name    9 non-null      object 
 6   units_of_measure  9 non-null      object 
 7   arithmetic_mean   9 non-null      float64
 8   aqi               9 non-null      int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 720.0+ bytes


**Formulate your null and alternative hypotheses here:**

*   $H_0$: The mean AQI of Michigan is less than or equal to 10.
*   $H_A$: The mean AQI of Michigan is greater than 10.


#### Significance Level (remains at 5%)

#### Determine the appropriate test procedure:

Here, you are comparing one sample mean relative to a particular value in one direction. Therefore, you will utilize a **one-sample  𝑡-test**. 

#### Compute the P-value

In [54]:
# Compute your p-value here
h3_result = stats.ttest_1samp(MI_air_quality['aqi'], 10, alternative='greater')
print(h3_result)
p_value_h3 = h3_result[1]
print(p_value_h3)

TtestResult(statistic=np.float64(-1.7395913343286131), pvalue=np.float64(0.9399405193140109), df=np.int64(8))
0.9399405193140109


#### **Question 4. What is your P-value for hypothesis 3, and what does this indicate for your null hypothesis?**

A: The p-value for hypothesis 3 is 0.9399, which is greatly larger than our significance level, 0.05, indicating that we cannot reject the null hypothesis and the mean AQI of Michigan is less than or equal to 10.

## Step 4. Results and Evaluation

Now that you've completed your statistical tests, you can consider your hypotheses and the results you gathered.

#### **Question 5. Did your results show that the AQI in Los Angeles County was statistically different from the rest of California?**

A: Yes, since we reject the null, the results show that the AQI in Los Angeles County was statistically different from the rest of California

#### **Question 6. Did New York or Ohio have a lower AQI?**

A: New York has a lower AQI, because we reject the null that New York is greater than or equal to Ohio.

#### **Question 7: Will Michigan be affected by the new policy impacting states with a mean AQI of 10 or greater?**



A: No, Michigan will not be affected by the new policy, because we cannot reject the null, showing Michigan mean AQI is below 10.

# Conclusion

**What are key takeaways from this project?**

A: When we conduct a test, we should be careful about the direction of the test, is it two-sided or one-sided. When we have 2 samples with 2 different distributions, we should use two-sample 2-test, however, if we only have 1 sample with 1 mean, we should use one-sample t-test. The p-value should be compared with our significance level, to help us draw the conclusion.

**What would you consider presenting to your manager as part of your findings?**

A: The 3 conclusions from the 3 hypothesis, which are:
1. The AQI in Los Angeles County was statistically different from the rest of California.
2. New York has lower AQI compared to Ohio.
3. Michigan will not be affected by the new policy.


**What would you convey to external readers?**

A: Government should look into the reason why Los Angeles county has different air quality compared to other counties, and see the direction of the difference, and take actions.
